# 🧹 Data Cleaning & Preprocessing

## Introduction

Following the Data Understanding phase, several data quality issues were identified that require preprocessing before performing exploratory data analysis and machine learning.

The objective of this notebook is to improve the quality and consistency of the dataset while preserving the integrity of the original information.

The preprocessing steps performed in this notebook include:

- Handling missing values
- Removing duplicate records
- Correcting data types
- Identifying unnecessary features
- Detecting data leakage
- Creating the target variable
- Saving a cleaned dataset for subsequent analysis

# 📚 Import Required Libraries

The following libraries are imported to perform data cleaning and preprocessing tasks.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [ ]:
from pathlib import Path


def find_project_root() -> Path:
    """
    Locate the project root whether the notebook is launched
    from the project root or from inside the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Run this notebook from inside the "
        "Flight_Delay_Prediction project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

RAW_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

# 📂 Load Dataset

The original dataset is loaded in preparation for preprocessing.

All cleaning operations performed in this notebook will be documented to ensure transparency and reproducibility.

In [ ]:
raw_data_path = (
    RAW_DATA_DIRECTORY
    / "flights_sample_3m.csv"
)

if not raw_data_path.exists():
    raise FileNotFoundError(
        f"Raw dataset was not found at:\n{raw_data_path}\n\n"
        "Place flights_sample_3m.csv inside the data/raw folder."
    )

df = pd.read_csv(raw_data_path)

print("Dataset loaded from:")
print(raw_data_path)

print("\nDataset shape:")
print(df.shape)

df.head()

# 🔍 Create a Copy of the Original Dataset

To preserve the raw dataset, a copy is created before applying any preprocessing operations.

This practice ensures that the original data remains unchanged and allows preprocessing steps to be reproduced if necessary.

In [ ]:
df_clean = df.copy()

# 📊 Initial Dataset Information

Before cleaning the data, it is useful to confirm the dataset dimensions and inspect the initial number of observations and variables.

In [ ]:
print(f"Rows : {df_clean.shape[0]:,}")
print(f"Columns : {df_clean.shape[1]}")

# 📋 Data Cleaning Strategy

Before modifying the dataset, each feature containing missing values is analyzed individually.

Different variables require different preprocessing techniques because missing values may have different meanings.

The following strategy will be applied:

| Situation | Action |
|------------|--------|
| Missing values represent unavailable information | Keep or encode appropriately |
| Missing values caused by cancelled flights | Investigate before removal |
| Variables that introduce data leakage | Remove before model training |
| Duplicate records | Remove if confirmed |
| Incorrect data types | Convert to appropriate format |

This approach ensures that preprocessing decisions are based on the business meaning of the data rather than applying a single technique to every feature.

# 🔍 3.1 Duplicate Records

Duplicate observations can introduce redundancy and bias into machine learning models.

Before cleaning the dataset, duplicate records are identified to determine whether they should be removed.

If duplicate records are found, they will be removed while keeping the first occurrence of each record.

In [ ]:
# Number of duplicate rows
duplicates = df_clean.duplicated().sum()

print(f"Duplicate Rows: {duplicates:,}")

### Interpretation

No duplicate records were found in the dataset.

Therefore, no duplicate removal was required, and all observations were retained for further preprocessing.

# 🕳️ 3.2 Missing Values Analysis

Missing values are common in real-world datasets and must be carefully analyzed before applying any preprocessing techniques.

Rather than immediately removing or imputing missing values, this section investigates the extent and pattern of missing data across all features.

Understanding the distribution and business meaning of missing values helps determine the most appropriate cleaning strategy for each variable.

In [ ]:
missing_values = pd.DataFrame({
    "Missing Values": df_clean.isnull().sum(),
    "Missing Percentage (%)": round(df_clean.isnull().mean() * 100, 2)
})

missing_values = missing_values.sort_values(
    by="Missing Values",
    ascending=False
)

missing_values

In [ ]:
missing_values[missing_values["Missing Values"] > 0]

### 📝 Interpretation

The missing values analysis revealed that the dataset contains missing observations in **17 out of the 32 features**.

Several important observations can be made:

- `CANCELLATION_CODE` contains **2,920,860 missing values (97.36%)**. This is expected because the cancellation reason is only recorded for cancelled flights.

- The delay-related variables (`DELAY_DUE_CARRIER`, `DELAY_DUE_WEATHER`, `DELAY_DUE_NAS`, `DELAY_DUE_SECURITY`, and `DELAY_DUE_LATE_AIRCRAFT`) each contain **82.20% missing values**. These values are expected because delay causes are only recorded when a qualifying delay occurs.

- Variables such as `DEP_TIME`, `DEP_DELAY`, `TAXI_OUT`, `WHEELS_OFF`, `WHEELS_ON`, `TAXI_IN`, `ARR_TIME`, `ARR_DELAY`, `ELAPSED_TIME`, and `AIR_TIME` contain approximately **2.6–2.9% missing values**. These missing values are likely associated with cancelled or diverted flights.

- `CRS_ELAPSED_TIME` contains only **14 missing values**, representing a negligible proportion of the dataset.

Overall, the missing values appear to have **business meaning** rather than resulting from data corruption. Therefore, each feature will be handled according to its role in the dataset instead of applying a single imputation strategy to all variables.

# 📋 3.3 Data Cleaning Decisions

Based on the missing values analysis, each feature will be treated according to its business meaning and relevance to the prediction task.

The following table summarizes the planned preprocessing strategy.

In [ ]:
cleaning_plan = pd.DataFrame({
    "Column": [
        "CANCELLATION_CODE",
        "DELAY_DUE_CARRIER",
        "DELAY_DUE_WEATHER",
        "DELAY_DUE_NAS",
        "DELAY_DUE_SECURITY",
        "DELAY_DUE_LATE_AIRCRAFT",
        "DEP_TIME",
        "DEP_DELAY",
        "TAXI_OUT",
        "WHEELS_OFF",
        "WHEELS_ON",
        "TAXI_IN",
        "ARR_TIME",
        "ARR_DELAY",
        "ELAPSED_TIME",
        "AIR_TIME",
        "CRS_ELAPSED_TIME"
    ],
    "Decision": [
        "Drop",
        "Drop",
        "Drop",
        "Drop",
        "Drop",
        "Drop",
        "Investigate",
        "Investigate",
        "Investigate",
        "Investigate",
        "Investigate",
        "Investigate",
        "Investigate",
        "Investigate",
        "Investigate",
        "Investigate",
        "Fill or Remove"
    ],
    "Reason": [
        "Only available for cancelled flights",
        "Post-flight information (data leakage)",
        "Post-flight information (data leakage)",
        "Post-flight information (data leakage)",
        "Post-flight information (data leakage)",
        "Post-flight information (data leakage)",
        "Likely missing due to cancelled/diverted flights",
        "Likely missing due to cancelled/diverted flights",
        "Likely missing due to cancelled/diverted flights",
        "Likely missing due to cancelled/diverted flights",
        "Likely missing due to cancelled/diverted flights",
        "Likely missing due to cancelled/diverted flights",
        "Likely missing due to cancelled/diverted flights",
        "Used to create target variable",
        "Missing mainly in cancelled/diverted flights",
        "Missing mainly in cancelled/diverted flights",
        "Only 14 missing values"
    ]
})

cleaning_plan

# 🔎 3.4 Investigating Missing Values

Before deciding how to handle the missing values, it is important to understand **why** they exist.

The flight dataset contains information about cancelled and diverted flights, which may explain the missing values observed in several operational variables.

This section investigates the relationship between missing values and flight cancellations or diversions to ensure that preprocessing decisions are based on evidence rather than assumptions.

In [ ]:
# Number of cancelled flights

cancelled = df_clean["CANCELLED"].value_counts()

cancelled

In [ ]:
# Number of diverted flights

diverted = df_clean["DIVERTED"].value_counts()

diverted

In [ ]:
# Missing ARR_DELAY for cancelled flights

cancelled_missing = df_clean[
    (df_clean["CANCELLED"] == 1) &
    (df_clean["ARR_DELAY"].isnull())
].shape[0]

print(f"Cancelled flights with missing ARR_DELAY: {cancelled_missing:,}")

In [ ]:
# Missing ARR_DELAY for diverted flights

diverted_missing = df_clean[
    (df_clean["DIVERTED"] == 1) &
    (df_clean["ARR_DELAY"].isnull())
].shape[0]

print(f"Diverted flights with missing ARR_DELAY: {diverted_missing:,}")

In [ ]:
remaining_missing = df_clean[
    (df_clean["ARR_DELAY"].isnull()) &
    (df_clean["CANCELLED"] == 0) &
    (df_clean["DIVERTED"] == 0)
]

print(f"Remaining unexplained missing values: {remaining_missing.shape[0]:,}")

### 📝 Interpretation

The investigation revealed that:

- **79,140 flights** were cancelled.
- **7,056 flights** were diverted.
- Both cancelled and diverted flights account for **86,196** observations.

The dataset contains **86,198 missing values** in the `ARR_DELAY` column. Therefore, **all but two** missing values can be explained by cancelled or diverted flights.

The remaining **2 observations** require further investigation because they are neither cancelled nor diverted but still have missing arrival delay information.

This finding confirms that the majority of missing operational values are associated with specific flight events rather than random data quality issues. Therefore, these observations will be handled carefully during preprocessing.

# 🔎 3.5 Investigating Unexplained Missing Values

Although almost all missing arrival delays are associated with cancelled or diverted flights, two observations remain unexplained.

These records are inspected individually to determine whether they contain additional inconsistencies or missing operational information.

Understanding these exceptional cases helps ensure that preprocessing decisions are fully justified.

In [ ]:
unexplained = df_clean[
    (df_clean["ARR_DELAY"].isnull()) &
    (df_clean["CANCELLED"] == 0) &
    (df_clean["DIVERTED"] == 0)
]

unexplained

### 📝 Interpretation

Two exceptional observations were identified where flights were neither cancelled nor diverted, yet all arrival-related information was missing.

Both records contain valid departure information (`DEP_TIME`, `DEP_DELAY`, and `WHEELS_OFF`) but lack `WHEELS_ON`, `ARR_TIME`, `ARR_DELAY`, `ELAPSED_TIME`, and `AIR_TIME`.

Since the target variable (`ARR_DELAY`) cannot be determined for these observations, these records are considered incomplete and will be removed from the dataset.

Removing these two records has a negligible impact on the dataset because they represent only **2 out of 3,000,000 observations**.

# 🗑️ 3.6 Remove Incomplete Flight Records

The investigation identified two incomplete flight records that were neither cancelled nor diverted but lacked all arrival-related information.

Since these observations do not contain the target variable required for machine learning, they will be removed from the dataset.

In [ ]:
# Remove the two incomplete records

df_clean = df_clean.drop(unexplained.index)

print(f"New Dataset Shape: {df_clean.shape}")

# 🕒 3.7 Data Type Conversion

Before proceeding with feature engineering and machine learning, the data types of the variables must be verified and corrected where necessary.

The dataset contains a date column (`FL_DATE`) that is currently stored as a text (object) data type. Converting this column to a proper datetime format enables efficient date-based analysis and feature engineering in later stages.

No other data type conversions are required at this stage because the remaining variables already have appropriate numerical or categorical data types.

In [ ]:
# Check current data types

df_clean.dtypes

### 📝 Interpretation

The dataset contains one date variable (`FL_DATE`) stored as an object data type. Since date-related operations cannot be efficiently performed on string values, this feature will be converted to the datetime format.

The remaining variables already have suitable numerical or categorical data types and therefore require no additional conversion.

In [ ]:
# Convert FL_DATE to datetime

df_clean["FL_DATE"] = pd.to_datetime(df_clean["FL_DATE"])

In [ ]:
df_clean["FL_DATE"].dtype

# Filter Flights for Arrival-Delay Prediction

Cancelled and diverted flights do not have a valid arrival-delay outcome because they did not complete the originally scheduled flight normally.

Since this project predicts significant arrival delay, only completed, non-diverted flights with a known `ARR_DELAY` value are retained.

In [ ]:
# Keep only completed flights with a valid arrival-delay value

df_clean = df_clean[
    (df_clean["CANCELLED"] == 0) &
    (df_clean["DIVERTED"] == 0) &
    (df_clean["ARR_DELAY"].notna())
].copy()

print(f"Dataset shape after filtering: {df_clean.shape}")

In [ ]:
# Remove the old target variable if it already exists
if "IS_DELAYED" in df_clean.columns:
    df_clean = df_clean.drop(columns=["IS_DELAYED"])

# 🎯 3.8 Target Variable Creation

Machine learning classification algorithms require a target variable representing the outcome to be predicted.

This project aims to predict whether a flight experiences a significant arrival delay.

Following the U.S. Bureau of Transportation Statistics definition, a flight is considered delayed if its arrival delay exceeds **15 minutes**.

Therefore, a new binary target variable named **IS_DELAYED** will be created from the `ARR_DELAY` feature.

- **1:** Arrival delay greater than 15 minutes.
- **0:** Arrival delay less than or equal to 15 minutes.

In [ ]:
# Create the binary target variable
# A flight is delayed when it arrives 15 minutes or more late.
df_clean["IS_DELAYED"] = (
    df_clean["ARR_DELAY"] >= 15
).astype("int8")

In [ ]:
# Distribution of the target variable

target_distribution = pd.DataFrame({
    "Count": df_clean["IS_DELAYED"].value_counts().sort_index(),
    "Percentage (%)": (
        df_clean["IS_DELAYED"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

target_distribution = target_distribution.rename(index={
    0: "On Time (<15 min)",
    1: "Delayed (≥15 min)"
})

target_distribution

# 💾 Save the Preprocessed Dataset

The dataset is saved after completing the initial preprocessing steps. This version will be used as the input for the Feature Engineering notebook.

In [ ]:
cleaned_output_path = (
    PROCESSED_DATA_DIRECTORY
    / "preprocessed_flight_data.csv"
)

df_clean.to_csv(
    cleaned_output_path,
    index=False
)

print("Cleaned dataset saved to:")
print(cleaned_output_path)

print("\nSaved dataset shape:")
print(df_clean.shape)

# 📌 Notebook Summary

In this notebook, the initial preprocessing of the flight dataset was completed.

The following tasks were performed:

- Loaded the original dataset.
- Created a working copy for preprocessing.
- Checked for duplicate records.
- Investigated missing values.
- Examined exceptional incomplete flight records.
- Removed two incomplete observations.
- Converted the flight date to the datetime format.
- Created the binary target variable (`IS_DELAYED`).
- Saved the cleaned dataset for the next phase of the project.

The dataset is now ready for **Feature Engineering**, where additional predictive variables will be created and data leakage will be addressed before model development.